1 - Implemente uma classe que permita selecionar o conjunto de treinamento e teste. Use o código abaixo para verificar se sua implementação está funcionando.

In [ ]:
import numpy as np

def create_matrix(n_rows, m_cols):
	matrix = np.zeros((n_rows, m_cols)) # Initialize with correct dimensions
	for i in range(n_rows):
		matrix[i,:] = i # Set all elements in row i to i
	return matrix


# Exemplo de uso da função:
n_rows = 10 # Número de linhas
m_cols = 4 # Número de colunas

print('Matriz X:')
X = create_matrix(n_rows, m_cols)
# Exibindo a matriz
for row in X:
	print(row)

print('Vetor y:')
y = create_matrix(n_rows, 1)
# Exibindo a matriz
for row in y:
	print(row)

Matriz X:
[0. 0. 0. 0.]
[1. 1. 1. 1.]
[2. 2. 2. 2.]
[3. 3. 3. 3.]
[4. 4. 4. 4.]
[5. 5. 5. 5.]
[6. 6. 6. 6.]
[7. 7. 7. 7.]
[8. 8. 8. 8.]
[9. 9. 9. 9.]
Vetor y:
[0.]
[1.]
[2.]
[3.]
[4.]
[5.]
[6.]
[7.]
[8.]
[9.]


In [ ]:
class TrainTest:
	def __init__(self, X, y, p):
		if not (0<p<1):
			raise ValueError("p must be between 0 and 1")
		self.X = X
		self.y = y
		self.p = p
	
	def split(self): # separar dados 
		np.random.seed(42)
		
		n_samples = self.X.shape[0] # numero de linhas da matriz x
		n_train = int(n_samples * self.p) # pegar o numero de amostras para o treino, dado o percentual p

		indices = np.arange(n_samples) #criar um array de indices de 0 a n_samples-1
		np.random.shuffle(indices) # embaralhar os indices (in-place)

		train_indices = indices[:n_train]
		test_indices = indices[n_train:]

		X_train = self.X[train_indices]
		y_train = self.y[train_indices]
		X_test = self.X[test_indices]
		y_test = self.y[test_indices]

		return X_train, y_train, X_test, y_test
	
splitter = TrainTest(X,y,0.8)
X_train, y_train, X_test, y_test = splitter.split()

print("\nX_train:")
for row in X_train:
	print(row)

print("\ny_train:")
for row in y_train:
	print(row)

print("\nX_test:")
for row in X_test:
	print(row)

print("\ny_test:")
for row in y_test:
	print(row)



X_train:
[8. 8. 8. 8.]
[1. 1. 1. 1.]
[5. 5. 5. 5.]
[0. 0. 0. 0.]
[7. 7. 7. 7.]
[2. 2. 2. 2.]
[9. 9. 9. 9.]
[4. 4. 4. 4.]

y_train:
[8.]
[1.]
[5.]
[0.]
[7.]
[2.]
[9.]
[4.]

X_test:
[3. 3. 3. 3.]
[6. 6. 6. 6.]

y_test:
[3.]
[6.]


2 - Consider o código abaixo que implementa o classificador k-vizinhos. Crie uma classe que permite realizar a classificação. Implemente o métodos predict, bem como outros disponíveis no scikit-learn: https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.KNeighborsClassifier.html

Os atributos deve ser X, y, k e a distância. O método deve ser o predict.

In [ ]:
import numpy as np
import statistics
from scipy.spatial import distance
from collections import Counter # Necessário para contar as classes no predict_proba

def knn(x_train, y_train, x_test, k):
	distances = [] #lista que armazena as distâncias
	x1 = x_test # elemento a ser classificado
	for x2 in x_train: # distancia entre a observacao elementos no conjunto de teste
		dist = distance.euclidean(x1,x2)
		distances.append(dist)
	indices = []
	cl = []
	for i in range(0,k):
		ind = np.argmin(distances) #elemento no conjunto de teste mais proximo
		#print('distance:', distances[ind],'index:', ind, 'class:', y_train[ind])
		distances[ind] = np.max(distances)
		indices.append(ind)
		cl.append(y_train[ind]) #guarda a classe
	print("Classes:",cl)
	classification = statistics.mode(cl)# encontra a classe
	return classification

In [115]:
class k_vizinhos:
	def __init__ (self, k = 3):
		self.k = k
		self.x_train = None
		self.y_train = None
		self.classes_ = None		
	
	def fit(self, x_train, y_train): # apenas armazena os dados
		self.x_train = np.array(x_train)
		self.y_train = np.array(y_train).ravel()
		self.classes_ = np.unique(self.y_train) # Armazena classes únicas, ordenadas
		return self

	def predict(self, x_test):
		
		if self.x_train is None or self.y_train is None:
			raise ValueError("Chame o método fit primeiro.")
	
		x_test = np.array(x_test) # Garantir que x_test seja um array numpy
		dists = distance.cdist(x_test, self.x_train, metric='euclidean') # Calculamos a matriz de distâncias entre todo o teste e todo o treino de uma vez
		
		predictions = []
		for row in dists:
			k_indices = np.argsort(row)[:self.k] # Índices dos k vizinhos mais próximos
			k_nearlest_labels = self.y_train[k_indices]

			prediction = statistics.mode(k_nearlest_labels) # Encontra a classe mais comum entre os vizinhos
			predictions.append(prediction)
		return np.array(predictions)
	
	def predict_proba(self, X_test):

		if self.x_train is None or self.y_train is None:
			raise ValueError("Chame o método 'fit' primeiro.")
			
		X_test = np.array(X_test)
		
		# Usamos o cdist aqui também para manter a eficiência
		dists = distance.cdist(X_test, self.x_train, metric='euclidean')
		
		all_probabilities = []
		for row in dists:
			# Pega as classes dos K vizinhos mais próximos
			k_indices = np.argsort(row)[:self.k]
			neighbors_labels = self.y_train[k_indices]
			
			# Conta as ocorrências de cada classe
			label_counts = Counter(neighbors_labels)
			
			# Inicializa as probabilidades com zero para todas as classes
			probabilities = np.zeros(len(self.classes_))
			
			# Preenche o array de acordo com a ordem de self.classes_
			for i, cls in enumerate(self.classes_):
				# Divide a contagem pelo número de vizinhos K (não n_neighbors)
				probabilities[i] = label_counts.get(cls, 0) / self.k 
			
			all_probabilities.append(probabilities)
			
		return np.array(all_probabilities)
	
	def score(self, X_test, y_test):
			y_pred = self.predict(X_test)
			y_true = np.array(y_test).ravel() # Achatamos o real para comparar
			return np.mean(y_pred == y_true)


In [118]:
splitter = TrainTest(X,y,0.8)
X_train, y_train, X_test, y_test = splitter.split()

knn_classifier = k_vizinhos(k=9)
knn_classifier.fit(X_train, y_train)
prediction = knn_classifier.predict(X_test)
predicao_proba = knn_classifier.predict_proba(X_test)

for i in range(len(X_test)):
	print(f"Teste: {X_test[i]}, Predição: {prediction[i]}, Classe Real: {y_test[i]}")

acuracia = knn_classifier.score(X_test, y_test)
print(f"Acurácia: {acuracia * 100:.2f}%")

print("\nProbabilidades de cada classe para cada amostra do conjunto de teste:")
for i in range(len(X_test)):
	print(f"Teste: {X_test[i]}, Probabilidades: {predicao_proba[i]}")

Teste: [3. 3. 3. 3.], Predição: 4.0, Classe Real: [3.]
Teste: [6. 6. 6. 6.], Predição: 5.0, Classe Real: [6.]
Acurácia: 0.00%

Probabilidades de cada classe para cada amostra do conjunto de teste:
Teste: [3. 3. 3. 3.], Probabilidades: [0.11111111 0.11111111 0.11111111 0.11111111 0.11111111 0.11111111
 0.11111111 0.11111111]
Teste: [6. 6. 6. 6.], Probabilidades: [0.11111111 0.11111111 0.11111111 0.11111111 0.11111111 0.11111111
 0.11111111 0.11111111]
